# 1.Data Preprocessing

In [ ]:
# Data Preparation:
#   Data cleaning
#   Feature engineering
#   Splitting the data into training and test sets
#Exploratory Data Analysis (EDA):
#   Demand, fare, and congestion patterns
#Modeling:
#   Time Series Forecasting (SARIMA, LSTM)
#   Regression Models (Linear Regression, Gradient Boosting)
#   Clustering (K-Means, DBSCAN)
#   Classification (Random Forest, Decision Trees)
#Evaluation and Prediction.
#   Mean Squared Error (MSE) for regression models
#   F1-score for classification tasks.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import csv
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:


# file paths for 8 months
import pandas as pd

file_paths = [
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-01.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-02.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-03.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-04.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-05.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-06.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-07.csv',
    '/content/drive/My Drive/Minor_AK8/yellow_tripdata_2021-08.csv',

]

# Columns you want to drop
columns_to_drop = ['VendorID', 'store_and_fwd_flag', 'payment_type', 'RatecodeID',
                   'extra', 'mta_tax', 'improvement_surcharge', 'tolls_amount', 'congestion_surcharge']

# Loop through each file, load it, drop columns, and save it back
dataframes = []  # To store each dataframe
for file_path in file_paths:
    # Loading the CSV into a dataframe
    df = pd.read_csv(file_path)

    # Dropping the specified columns
    #df = df.drop(columns=columns_to_drop)

    # Save the modified dataframe back to the same file
    df.to_csv(file_path, index=False)

    # Appending the dataframe to the list
    dataframes.append(df)

# concatenate all the dataframes into one
all_data = pd.concat(dataframes)






<ipython-input-4-bfe38c4a91ec>:24: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
<ipython-input-4-bfe38c4a91ec>:24: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
<ipython-input-4-bfe38c4a91ec>:24: DtypeWarning: Columns (2,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
<ipython-input-4-bfe38c4a91ec>:24: DtypeWarning: Columns (2,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
<ipython-input-4-bfe38c4a91ec>:24: DtypeWarning: Columns (2,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
<ipython-input-4-bfe38c4a91ec>:24: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [ ]:
all_data.head(10)

In [ ]:
# converting pickup and dropoff times to datetime

In [ ]:
all_data['tpep_pickup_datetime'] = pd.to_datetime(all_data['tpep_pickup_datetime'])
all_data['tpep_dropoff_datetime'] = pd.to_datetime(all_data['tpep_dropoff_datetime'])

In [ ]:
# Feature extraction: Trip duration (in minutes)

In [ ]:
from re import A
all_data['tpep_trip_duration'] = (all_data['tpep_dropoff_datetime'] - all_data['tpep_pickup_datetime']).dt.total_seconds() / 60

In [ ]:
# Feature extraction: Hour, day, and month

In [ ]:
all_data['hour'] = all_data['tpep_pickup_datetime'].dt.hour
all_data['day_of_week'] = all_data['tpep_pickup_datetime'].dt.dayofweek
all_data['month'] = all_data['tpep_pickup_datetime'].dt.month

In [ ]:
all_data.tail(10)

# 2. Exploratory Data Analysis (EDA)
#    Plot demand, fare, and congestion patterns based on time features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Demand pattern by hour

In [ ]:
sns.lineplot(x='hour', y='tpep_trip_duration', data=all_data)
plt.title('Demand Pattern by Hour')
plt.show()

In [ ]:
# Fare pattern by day of the week

In [ ]:
# Calculate the average fare amount for each day of the week
# This will group the data by 'day_of_week' and calculate the mean of 'fare_amount' for each group.
# The reset_index() is used to convert the grouped result back into a DataFrame.

In [ ]:
daily_avg_fare = all_data.groupby('day_of_week')['fare_amount'].mean().reset_index()

In [ ]:
# Now plot the average fare by day of the week using the aggregated data


In [ ]:
sns.lineplot(x='day_of_week', y='fare_amount', data=daily_avg_fare)
plt.title('Average Fare by Day of the Week')
plt.show()

In [ ]:
# Check correlation between features

In [ ]:
chunk_size = 50000

chunk_correlations = []
total_rows = len(all_data)

# Process data in chunks
for i in range(0, total_rows, chunk_size):
    chunk = all_data[i:i + chunk_size]  # Extract a chunk of data
    # Replace '\\N' with NaN and convert to numeric within the chunk
    for column in chunk.columns:
        if chunk[column].astype(str).str.contains(r'\\N').any():
            chunk[column] = pd.to_numeric(chunk[column].astype(str).str.replace(r'\\N', 'NaN'), errors='coerce')

    # Calculate correlation for the current chunk and append to the list
    chunk_correlations.append(chunk.corr())

# Concatenate the correlations from all chunks
all_correlations = pd.concat(chunk_correlations)

# Calculate the average correlation across all chunks
average_correlation = all_correlations.groupby(all_correlations.index).mean()
plt.figure(figsize=(10,6))
sns.heatmap(average_correlation, annot=True)
plt.show()



#  3. Time Series Forecasting (SARIMA): forecasting trip demand during peak seasons or events.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

In [ ]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

In [ ]:
df_filtered = df[(df['tpep_pickup_datetime'] >= '2021-01-01') & (df['tpep_pickup_datetime'] <= '2021-08-30')]


In [ ]:
# Convert 'tpep_dropoff_datetime' to datetime objects before subtraction
df_filtered['tpep_dropoff_datetime'] = pd.to_datetime(df_filtered['tpep_dropoff_datetime'])

# Calculate trip duration
df_filtered['tpep_trip_duration'] = (df_filtered['tpep_dropoff_datetime'] - df_filtered['tpep_pickup_datetime']).dt.total_seconds()

In [ ]:
df_filtered.head(10)

In [ ]:
daily_demand = df_filtered.resample('D', on='tpep_pickup_datetime')['tpep_trip_duration'].count()


In [ ]:
# Split the data (80% for training, 20% for testing)
train_size = int(0.8 * len(daily_demand))
train, test = daily_demand[:train_size], daily_demand[train_size:]


In [ ]:
# Install the pmdarima package for auto_arima
!pip install pmdarima

from pmdarima import auto_arima

# Auto ARIMA to determine the best parameters for the model
auto_model = auto_arima(train, seasonal=True, m=7, trace=True, suppress_warnings=True)

# Print the best model summary
print(auto_model.summary())


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Fit the best SARIMA model
sarima_model = SARIMAX(train,
                       order=(3,2,1),              # (p, d, q)
                       seasonal_order=(1,1,1,7))  # (P, D, Q, m)

sarima_results = sarima_model.fit()

# Print the summary of the model
print(sarima_results.summary())


In [ ]:
# Make predictions for the test set
predictions = sarima_results.get_forecast(steps=len(test))
forecast = predictions.predicted_mean
conf_int = predictions.conf_int()  # Confidence intervals for predictions


In [ ]:
import matplotlib.pyplot as plt

# Plot true demand vs forecasted demand
plt.figure(figsize=(10,6))
plt.plot(test.index, test, label='True Demand', color='blue')
plt.plot(test.index, forecast, label='Forecasted Demand', color='red')

# Optionally plot the confidence intervals
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='pink', alpha=0.3)

plt.title('Demand Forecasting (Jan-Aug 2021) using SARIMA')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.show()


In [ ]:

from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate MSE and MAE
mse = mean_squared_error(test, forecast)
mae = mean_absolute_error(test, forecast)

print(f'Mean Squared Error: {mse}')
print(f'Mean Absolute Error: {mae}')


#4. LSTM Model for Time Series Forecasting

In [ ]:
# Resample the data to daily frequency, counting the number of trips per day
daily_demand = df_filtered.resample('D', on='tpep_pickup_datetime')['tpep_trip_duration'].count()


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Reshape the data for scaling
daily_demand = daily_demand.values.reshape(-1, 1)

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(daily_demand)


In [ ]:
# Create sequences for LSTM
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps), 0])
        y.append(data[i + time_steps, 0])
    return np.array(X), np.array(y)

# Define the number of time steps (e.g., using the past 30 days to predict the next day)
time_steps = 30

# Create sequences
X, y = create_sequences(scaled_data, time_steps)

# Reshape X to [samples, time steps, features] for LSTM
X = X.reshape(X.shape[0], X.shape[1], 1)


In [ ]:
# Define training size (80% of the data)
train_size = int(0.8 * len(X))

# Create train and test sets
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Build the LSTM model
model = Sequential()

# First LSTM layer
model.add(LSTM(units=100, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))

# Second LSTM layer
model.add(LSTM(units=100, return_sequences=False))
model.add(Dropout(0.2))

# Output layer
model.add(Dense(units=1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))


In [ ]:
# Make predictions on the test set
predictions = model.predict(X_test)

# Inverse the scaling to get actual demand values
predictions = scaler.inverse_transform(predictions.reshape(-1, 1))
y_test_scaled = scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:

# Plot actual vs predicted demand
plt.figure(figsize=(10,6))
plt.plot(y_test_scaled, label='True Demand', color='blue')
plt.plot(predictions, label='Predicted Demand', color='red')

plt.title('Demand Forecasting using LSTM (Jan-Aug 2021)')
plt.xlabel('Time')
plt.ylabel('Demand')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate MSE and MAE
mse = mean_squared_error(y_test_scaled, predictions)
mae = mean_absolute_error(y_test_scaled, predictions)

print(f'Mean Squared Error: {mse}')
print(f'Mean Absolute Error: {mae}')


# 5. Linear Regression for Fare Prediction

In [ ]:
import pandas as pd



# Convert 'pickup_datetime' to datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Filter data for the period Jan 2021 to Aug 2021
df_filtered = df[(df['tpep_pickup_datetime'] >= '2021-01-01') & (df['tpep_pickup_datetime'] <= '2021-08-31')]


In [ ]:
# Convert 'tpep_dropoff_datetime' to datetime objects before subtraction
df_filtered['tpep_dropoff_datetime'] = pd.to_datetime(df_filtered['tpep_dropoff_datetime'])

# Calculate trip duration
df_filtered['tpep_trip_duration'] = (df_filtered['tpep_dropoff_datetime'] - df_filtered['tpep_pickup_datetime']).dt.total_seconds()

In [ ]:
# Extract hour, day, and weekday information from pickup time
df_filtered['hour'] = df_filtered['tpep_pickup_datetime'].dt.hour
df_filtered['day'] = df_filtered['tpep_pickup_datetime'].dt.day
df_filtered['weekday'] = df_filtered['tpep_pickup_datetime'].dt.weekday

# Select the relevant features
features = df_filtered[['tpep_trip_duration', 'hour', 'weekday', 'passenger_count']]
target = df_filtered['fare_amount']  # Assuming 'fare_amount' is the column name for fare


In [ ]:
# Check for missing values
print(features.isnull().sum())
df_filtered = df_filtered.dropna(subset=['fare_amount'])

# Recalculate features and target after dropping missing values
features = df_filtered[['tpep_trip_duration', 'hour', 'weekday', 'passenger_count']]
target = df_filtered['fare_amount']

# Drop any rows with invalid values (e.g., negative fare or duration)
#features = features[features['tpep_trip_duration'] > 0]
#target = target[features.index]


In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)



In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)


In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)


In [ ]:
import matplotlib.pyplot as plt

# Plot actual vs predicted fares
plt.figure(figsize=(10,6))
plt.scatter(y_test, y_pred, alpha=0.3, color='blue')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2)

plt.title('Actual vs Predicted Fares (Linear Regression)')
plt.xlabel('Actual Fare')
plt.ylabel('Predicted Fare')
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate MSE and MAE
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'Mean Absolute Error: {mae}')


# 6. gradient boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
# Convert 'pickup_datetime' to datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Filter data for the period Jan 2021 to Aug 2021
df_filtered = df[(df['tpep_pickup_datetime'] >= '2021-01-01') & (df['tpep_pickup_datetime'] <= '2021-08-31')]

In [ ]:
# Convert 'tpep_dropoff_datetime' to datetime objects before subtraction
df_filtered['tpep_dropoff_datetime'] = pd.to_datetime(df_filtered['tpep_dropoff_datetime'])

# Calculate trip duration
df_filtered['tpep_trip_duration'] = (df_filtered['tpep_dropoff_datetime'] - df_filtered['tpep_pickup_datetime']).dt.total_seconds()

In [ ]:
# Extract hour, day, and weekday information from pickup time
df_filtered['hour'] = df_filtered['tpep_pickup_datetime'].dt.hour
df_filtered['day'] = df_filtered['tpep_pickup_datetime'].dt.day
df_filtered['weekday'] = df_filtered['tpep_pickup_datetime'].dt.weekday

# Select the relevant features
features = df_filtered[['tpep_trip_duration', 'hour', 'weekday', 'passenger_count']]
target = df_filtered['fare_amount']  # Assuming 'fare_amount' is the column name for fare





In [ ]:
print(target.isnull().sum())

# Drop rows with missing values in the target variable before splitting
df_filtered = df_filtered.dropna(subset=['fare_amount'])

# Recalculate features and target after dropping missing values
features = df_filtered[['tpep_trip_duration', 'hour', 'weekday', 'passenger_count']]
target = df_filtered['fare_amount']


In [ ]:
# Define the hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of boosting stages
    'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
    'max_depth': [3, 4, 5],  # Maximum depth of individual estimators
    'min_samples_split': [2, 5, 10],  # Minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4],  # Minimum samples required in a leaf node
    'subsample': [0.8, 1.0],  # Fraction of samples used for each tree
    'max_features': ['auto', 'sqrt', 'log2']  # Number of features to consider when splitting
}


In [ ]:
# Initialize the model
#gb_model = GradientBoostingRegressor(random_state=42)


In [ ]:
# Perform grid search with 3-fold cross-validation
#grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)

# Fit the grid search to the training data
#grid_search.fit(X_train, y_train)


In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Initialize the Gradient Boosting Regressor
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)

# Train the model
model.fit(X_train, y_train)


In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)


In [ ]:
import matplotlib.pyplot as plt

# Plot actual vs predicted fares
plt.figure(figsize=(10,6))
plt.scatter(y_test, y_pred, alpha=0.3, color='blue')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2)

plt.title('Actual vs Predicted Fares (Gradient Boosting)')
plt.xlabel('Actual Fare')
plt.ylabel('Predicted Fare')
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate MSE and MAE
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'Mean Absolute Error: {mae}')


# 7.K-Means Clustering for Demand Zones

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
# Convert 'pickup_datetime' to datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Filter data for the period Jan 2021 to Aug 2021
df_filtered = df[(df['tpep_pickup_datetime'] >= '2021-01-01') & (df['tpep_pickup_datetime'] <= '2021-08-31')]

In [ ]:
df_filtered = df[['PULocationID']].dropna()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize the distribution of pickup locations (taxi zones)
plt.figure(figsize=(12, 6))
sns.histplot(df_filtered['PULocationID'], bins=100)
plt.title('Distribution of Pickup Zones (Taxi Zone IDs)')
plt.xlabel('Taxi Zone ID')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Convert PULocationID to integer (if not already)
df_filtered['PULocationID'] = df_filtered['PULocationID'].astype(int)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
# Count pickups per zone (demand)
zone_demand = df_filtered['PULocationID'].value_counts().reset_index()
zone_demand.columns = ['PULocationID', 'demand']


# Define the number of clusters (demand zones)
num_clusters = 20 # You can change this to experiment with different numbers of zones

# Initialize K-Means model
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

# Fit the model to the pickup location IDs (reshaped for clustering)
kmeans.fit(df_filtered[['PULocationID']])

# Get cluster labels for each taxi zone (pickup zone)
df_filtered['demand_zone'] = kmeans.labels_

# Get the centers of the clusters (centroids of demand zones)
cluster_centers = kmeans.cluster_centers_
print("Cluster Centers (Taxi Zone IDs):\n", cluster_centers)


In [ ]:
# Plot the demand zones (clusters)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df_filtered['PULocationID'], y=df_filtered['demand_zone'], hue=df_filtered['demand_zone'], palette='viridis')
plt.title('Demand Zones Based on K-Means Clustering of Pickup Locations')
plt.xlabel('Taxi Zone ID')
plt.ylabel('Demand Zone')
plt.show()


In [ ]:
# Apply K-Means clustering
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
zone_demand['demand_zone'] = kmeans.fit_predict(zone_demand[['demand']])

# Print cluster centers (demand levels in each zone)
print("Cluster Centers (Demand Levels):\n", kmeans.cluster_centers_)

In [ ]:
# Count the number of pickups in each demand zone
zone_counts = df_filtered['demand_zone'].value_counts().sort_index()

# Display demand per zone
for zone, count in zone_counts.items():
    print(f"Demand Zone {zone}: {count} pickups")


In [ ]:
# Add additional features for clustering
df['hour'] = pd.to_datetime(df['tpep_pickup_datetime']).dt.hour
df_filtered = df[['PULocationID', 'trip_distance', 'fare_amount', 'hour']].dropna()

# Apply clustering on multiple features
kmeans.fit(df_filtered)


In [ ]:
!pip install pyogrio

In [ ]:
import geopandas as gpd
import os

shapefile_path = '/content/drive/My Drive/Minor_AK8/geo_export_4d59f0fc-9556-415c-afe2-01d647788b58.shp'
#gpd.io.file.fiona_env.set_config_option("SHAPE_RESTORE_SHX", "YES")
# Check if the .shx file exists
shx_path = shapefile_path.replace('.shp', '.shx')
if not os.path.exists(shx_path):
    print(f"Warning: .shx file not found at {shx_path}. Attempting to restore...")
    # For older geopandas versions, try creating a .shx file
    # using a geospatial library like Fiona or pyshp if available
    # or consider updating your geopandas library.
    # Example using Fiona:
    # import fiona
    # with fiona.open(shapefile_path) as src:
    #     # This might trigger creation of .shx if missing
    #     pass

# Read the shapefile
taxi_zones = gpd.read_file(shapefile_path, engine='pyogrio')

In [ ]:
print(taxi_zones.columns)


In [ ]:
import geopandas as gpd

# Load the NYC Taxi Zones shapefile (adjust the path to your shapefile location)
shapefile_path = '/content/drive/My Drive/Minor_AK8/geo_export_4d59f0fc-9556-415c-afe2-01d647788b58.shp'
taxi_zones = gpd.read_file(shapefile_path)

# Ensure the column name is correct (replace 'location_i' with the correct one, such as 'LocationID')
taxi_zones['location_i'] = taxi_zones['location_i'].astype(int)

# Merge based on the correct column name
merged_data = taxi_zones.merge(zone_demand, left_on= 'location_i', right_on='PULocationID', how='left')
# Instead of using 'inplace=True', directly assign the result back to the column
merged_data['demand_zone'] = merged_data['demand_zone'].fillna(-1)  # -1 for zones without data


In [ ]:
import matplotlib.pyplot as plt

# Set up the figure
plt.figure(figsize=(20, 12))

# Plot the demand zones (color-coded by cluster/demand zone)
merged_data.plot(column='demand_zone', cmap='viridis', legend=True, linewidth=0.8)

# Add a title and labels
plt.title('NYC Taxi Demand Zones Based on K-Means Clustering', fontsize=15)
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.show()


# 8.  Classification: Random Forest for Congestion Prediction

In [ ]:
# Convert 'pickup_datetime' to datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Filter data for the period Jan 2021 to Aug 2021
df_filtered = df[(df['tpep_pickup_datetime'] >= '2021-01-01') & (df['tpep_pickup_datetime'] <= '2021-08-31')]

In [ ]:
# Select relevant columns for congestion prediction
# We'll use pickup time, location, dropoff time, etc.
df_filtered = df[['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'passenger_count']].dropna()

# Convert pickup and dropoff times to datetime format
df_filtered['tpep_pickup_datetime'] = pd.to_datetime(df_filtered['tpep_pickup_datetime'])
df_filtered['tpep_dropoff_datetime'] = pd.to_datetime(df_filtered['tpep_dropoff_datetime'])

# Extract useful time-related features (hour of day, day of week)
df_filtered['pickup_hour'] = df_filtered['tpep_pickup_datetime'].dt.hour
df_filtered['pickup_day'] = df_filtered['tpep_pickup_datetime'].dt.dayofweek

# Calculate trip duration
df_filtered['tpep_trip_duration'] = (df_filtered['tpep_dropoff_datetime'] - df_filtered['tpep_pickup_datetime']).dt.total_seconds() / 60.0

In [ ]:
# Define congestion label based on trip duration per mile
df_filtered['congestion'] = (df_filtered['tpep_trip_duration'] / df_filtered['trip_distance']) > df_filtered['tpep_trip_duration'].mean()

# Convert to binary labels: 1 for high congestion, 0 for low congestion
df_filtered['congestion'] = df_filtered['congestion'].astype(int)


In [ ]:

# Select features and target
X = df_filtered[['PULocationID', 'DOLocationID', 'trip_distance', 'passenger_count', 'pickup_hour', 'pickup_day']]
y = df_filtered['congestion']

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
rf_classifier.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict on the test set
y_pred = rf_classifier.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

# Display the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(conf_matrix)

# Display classification report
class_report = classification_report(y_test, y_pred)
print('Classification Report:')
print(class_report)

In [ ]:
import matplotlib.pyplot as plt

# Count of congestion classes
congestion_counts = df_filtered['congestion'].value_counts()

# Bar plot for congestion counts
plt.figure(figsize=(8, 5))
plt.bar(congestion_counts.index, congestion_counts.values, color=['skyblue', 'salmon'])
plt.xticks([0, 1], ['Not Congested', 'Congested'])
plt.title('Count of Congested vs. Not Congested Trips')
plt.xlabel('Congestion Status')
plt.ylabel('Number of Trips')
plt.show()


In [ ]:
import seaborn as sns

# Create a heatmap for pickup locations
plt.figure(figsize=(10, 6))
sns.kdeplot(
    x=df_filtered['PULocationID'],
    y=taxi_zones['location_i'],
    cmap='Reds',
    fill=True,
    thresh=0,
    levels=100,
    alpha=0.5
)
plt.title('Heatmap of Pickup Locations')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


In [ ]:
import folium

# Create a base map
map_center = [df_filtered['PULocationID'].mean(), taxi_zones['location_i'].mean()]
map_congestion = folium.Map(location=map_center, zoom_start=12)

# Add congestion points to the map
for lat, lon, congestion in zip(df_filtered['PULocationID'],taxi_zones['location_i'], df['congestion']):
    color = 'green' if congestion == 0 else 'red'
    folium.CircleMarker(
        location=[lat, lon],
        radius=2,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6
    ).add_to(map_congestion)

# Show the map
map_congestion.save('congestion_map.html')


In [ ]:
# Convert pickup datetime to pandas datetime if you have a date column
# df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Resample data by hour to calculate congestion
hourly_congestion = df_filtered.resample('H', on='tpep_pickup_datetime').mean()['congestion']

# Time series plot
plt.figure(figsize=(12, 6))
plt.plot(hourly_congestion.index, hourly_congestion, color='purple')
plt.title('Average Congestion Over Time')
plt.xlabel('Time')
plt.ylabel('Average Congestion')
plt.xticks(rotation=45)
plt.show()


In [ ]:
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [ ]:
# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': randint(50, 300),         # Number of trees
    'max_depth': [None] + list(range(5, 21)),  # Max depth of the trees
    'min_samples_split': randint(2, 10),      # Minimum samples required to split an internal node
    'min_samples_leaf': randint(1, 10),       # Minimum samples required to be at a leaf node
    'max_features': ['auto', 'sqrt']          # Number of features to consider when looking for the best split
}


In [ ]:
# Initialize Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)

# Set up RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf_classifier,
                                   param_distributions=param_dist,
                                   n_iter=100,        # Number of parameter settings sampled
                                   cv=3,              # Number of cross-validation folds
                                   n_jobs=-1,         # Use all available cores
                                   verbose=2,         # Verbosity level
                                   random_state=42)


In [ ]:
# Fit the random search to the training data
#random_search.fit(X_train, y_train)

# Best parameters found by RandomizedSearchCV
#print("Best Hyperparameters:", random_search.best_params_)


In [ ]:
# Use the best estimator to make predictions
#best_model = random_search.best_estimator_
# copy cell above confusion matrix


In [ ]:
!pip install pyngrok
!pip install streamlit




In [ ]:
!ngrok authtoken YOUR_AUTHTOKEN

In [ ]:
from pyngrok import ngrok

# ... other imports and code ...

ngrok.set_auth_token("2p3Qx5sHnpV6FnLxAd8tVShyzwN_3Yj93aBvaSyTQYcEBPosf")  # Replace YOUR_AUTHTOKEN with your actual authtoken

# Run Flask app on a specific port
port = 5000
public_url = ngrok.connect(port)

# Start the Flask app
app.run(port=port)
